# 2. Azure OpenAI

## 1. What is Azure OpenAI?

**Azure OpenAI** is Microsoft's managed Azure service for accessing OpenAI models and building enterprise AI applications.

It provides the model inference capability of OpenAI together with Azure capabilities such as:

- Microsoft Entra ID authentication
- RBAC
- Managed Identity
- Azure networking
- Monitoring
- Enterprise governance
- Integration with other Azure services

For new applications, Microsoft currently recommends the **Responses API**; Chat Completions remains available, particularly for existing integrations. 

---

# 2. Basic Architecture

```text
User
  ↓
Application
  ↓
Azure OpenAI Resource
  ↓
Model Deployment
  ↓
LLM
  ↓
Response
```

For an Agentic AI application:

```text
User
  ↓
FastAPI / Application
  ↓
LangGraph / Agent
  ↓
Azure OpenAI
  ↓
Tool Calling
  ├── REST API
  ├── Database
  ├── Azure AI Search
  └── Enterprise Systems
```

---

# 3. Azure OpenAI Resource

An **Azure OpenAI resource** is the Azure resource through which your application accesses the Azure OpenAI service.

Conceptually:

```text
Azure Subscription
       ↓
Resource Group
       ↓
Azure OpenAI Resource
       ↓
Model Deployment
```

The resource provides the endpoint and security boundary for your application.

---

# 4. Model Deployment ⭐⭐⭐⭐⭐

This is an important Azure-specific concept.

In Azure OpenAI, you generally create a **deployment** for the model you want to use.

```text
Azure OpenAI Resource
        │
        ├── gpt-deployment
        │
        └── embedding-deployment
```

Your application typically sends the **deployment name** when making inference requests.

Microsoft's current examples explicitly use:

```python
model="YOUR-DEPLOYMENT-NAME"
```

rather than directly passing the underlying model ID. 

### Interview Question

**Q: What is the difference between a model and deployment?**

**Answer:**

> "The model is the underlying foundation model, while the deployment is the configured endpoint through which my application accesses that model in Azure. In application code, I typically reference the deployment name."

---

# 5. Models

Azure OpenAI provides access to different OpenAI model families depending on availability, region, quota, and current Azure offering.

Typical categories include:

| Model Type | Usage |
|---|---|
| GPT models | Chat, reasoning, generation |
| Smaller GPT variants | Cost/latency-sensitive workloads |
| Embedding models | Semantic search and RAG |
| Multimodal models | Text + image scenarios |
| Reasoning models | Complex reasoning tasks |

**Interview tip:** Don't memorize a fixed model list because Azure model availability changes by region and over time.

---

# 6. How Do We Call Azure OpenAI?

There are two important authentication approaches.

### API Key

```text
Application
    ↓
API Key
    ↓
Azure OpenAI
```

### Microsoft Entra ID ⭐⭐⭐⭐⭐

```text
Application
    ↓
Managed Identity / Entra Credential
    ↓
Microsoft Entra ID
    ↓
Azure OpenAI
```

Microsoft documents both API-key and Entra ID approaches for Azure OpenAI. 

For enterprise production applications, **keyless authentication using Entra ID/Managed Identity** is generally the stronger architecture because application secrets don't need to be embedded or stored as API keys.

---

# 7. Simple Python Example

Current Azure documentation supports using the OpenAI Python SDK with the Azure OpenAI v1 endpoint. 

### API Key Example

```python
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url="https://YOUR-RESOURCE-NAME.openai.azure.com/openai/v1/"
)

response = client.chat.completions.create(
    model="YOUR-DEPLOYMENT-NAME",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful AI assistant."
        },
        {
            "role": "user",
            "content": "Explain RAG in simple terms."
        }
    ]
)

print(response.choices[0].message.content)
```

The important Azure-specific part is:

```python
base_url="https://YOUR-RESOURCE-NAME.openai.azure.com/openai/v1/"
```

and:

```python
model="YOUR-DEPLOYMENT-NAME"
```

rather than simply assuming the model name is the deployment name. 

---

# 8. Entra ID / Managed Identity Example

For enterprise applications, you can use `DefaultAzureCredential`.

```python
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://ai.azure.com/.default"
)

client = OpenAI(
    base_url="https://YOUR-RESOURCE-NAME.openai.azure.com/openai/v1/",
    api_key=token_provider
)

response = client.chat.completions.create(
    model="YOUR-DEPLOYMENT-NAME",
    messages=[
        {
            "role": "user",
            "content": "What is Agentic AI?"
        }
    ]
)

print(response.choices[0].message.content)
```

Microsoft's current documentation shows this Entra ID pattern using `DefaultAzureCredential` and a bearer-token provider. 

---

# 9. Chat Completions

Traditional Chat Completions uses messages:

```python
messages = [
    {
        "role": "system",
        "content": "You are an HR assistant."
    },
    {
        "role": "user",
        "content": "How many leave days do I have?"
    }
]
```

The model processes the conversation and returns an assistant message.

Microsoft currently describes **Responses API as the recommended approach for new applications**, while Chat Completions remains supported. 

---

# 10. Responses API

The Responses API is Microsoft's newer API surface for Azure OpenAI.

It supports capabilities important for modern AI applications, including:

- Multi-turn/stateful interactions
- Streaming
- Function calling
- File input
- Image input
- Hosted tools
- MCP-related capabilities
- Background tasks

Microsoft documents the Responses API as the recommended API for new Azure OpenAI applications. 

Conceptually:

```text
User
 ↓
Responses API
 ↓
Model
 ↓
Response
```

For an Agentic AI system:

```text
User
 ↓
Responses API
 ↓
Model
 ↓
Tool Call
 ↓
Tool Execution
 ↓
Tool Result
 ↓
Model
 ↓
Final Response
```

---

# 11. System / User / Assistant Messages

For Chat Completions:

| Role | Purpose |
|---|---|
| `system` | Defines behavior/persona/instructions |
| `user` | User's request |
| `assistant` | Previous model response |
| `tool` | Result returned from a tool |

Example:

```python
messages = [
    {
        "role": "system",
        "content": "You are an enterprise HR assistant."
    },
    {
        "role": "user",
        "content": "What is the leave policy?"
    }
]
```

---

# 12. Temperature

Temperature controls output variability.

```text
Low temperature
      ↓
More deterministic
```

```text
High temperature
      ↓
More diverse/creative
```

Typical enterprise RAG:

```text
temperature ≈ 0 - 0.3
```

For creative applications, you may use a higher value.

**Important:** Temperature is not a direct hallucination-control mechanism. Retrieval quality, grounding, prompts, model choice, validation, and guardrails also matter.

---

# 13. Token Management

An Azure OpenAI request contains:

```text
Input tokens
     +
Output tokens
     =
Total token usage
```

Example:

```text
System prompt      → 100 tokens
User question      → 50 tokens
RAG context        → 1,000 tokens
                    ─────────────
Input              → 1,150 tokens

Generated answer   → 300 tokens
```

Total ≈ **1,450 tokens**.

Token management matters for:

- Cost
- Latency
- Context limits
- RAG quality

---

# 14. Streaming ⭐⭐⭐⭐⭐

Instead of waiting for the entire answer:

```text
User
 ↓
Azure OpenAI
 ↓
"RAG..."
"RAG is..."
"RAG is a..."
...
```

The application receives partial output progressively.

This is useful for:

- Chatbots
- AI assistants
- Interactive agents
- ChatGPT-style interfaces

The current Responses API supports streaming through server-sent events. 

---

# 15. Function / Tool Calling ⭐⭐⭐⭐⭐

This is particularly relevant to your **Agentic AI JD**.

Suppose the user asks:

> "What is my leave balance?"

The model determines that it needs a tool:

```text
User
 ↓
Azure OpenAI
 ↓
Tool Call
 ↓
get_leave_balance(employee_id)
 ↓
HR API
 ↓
Tool Result
 ↓
Azure OpenAI
 ↓
Final Answer
```

The LLM **does not directly execute your backend function**. Your application receives the tool call, executes the function/API, provides the result back to the model, and then the model generates the final response.

---

# 16. Structured Output

For enterprise AI applications, you often don't want:

```text
"Here is the extracted information..."
```

You want predictable JSON:

```json
{
  "employee_name": "Suraj",
  "leave_type": "casual",
  "days": 2
}
```

This is useful for:

- API payloads
- Database insertion
- Workflow automation
- Agent state
- Downstream services

For your **AutoShift-style workflow**, structured output is particularly useful because the LLM output can be validated with **Pydantic** before calling the downstream API.

---

# 17. Azure OpenAI + RAG

Azure OpenAI is the **generation/model layer**, not your complete RAG system.

```text
Documents
    ↓
Chunking
    ↓
Embeddings
    ↓
Azure AI Search
    ↓
Relevant Chunks
    ↓
Prompt + Context
    ↓
Azure OpenAI
    ↓
Grounded Answer
```

This is where **Azure AI Search** comes into the architecture.

---

# 18. Azure OpenAI + LangChain

You can use Azure OpenAI as the LLM inside LangChain.

```text
User
 ↓
LangChain
 ↓
Prompt
 ↓
Azure OpenAI
 ↓
Response
```

For RAG:

```text
User
 ↓
LangChain
 ↓
Retriever
 ↓
Azure AI Search
 ↓
Context
 ↓
Azure OpenAI
 ↓
Answer
```

For Agentic AI:

```text
User
 ↓
LangGraph
 ↓
Azure OpenAI
 ↓
Tool Selection
 ├── Search
 ├── REST API
 ├── Database
 └── Enterprise Service
```

---

# 19. Azure OpenAI vs AWS Bedrock

Since your strongest production experience is **AWS Bedrock + Claude Sonnet**, this comparison is critical.

| Azure OpenAI | AWS Bedrock |
|---|---|
| Azure managed AI service | AWS managed foundation-model service |
| OpenAI model ecosystem | Multi-provider model ecosystem |
| GPT models | Claude, Llama, Titan, etc. |
| Microsoft Entra ID | AWS IAM |
| Managed Identity | IAM Roles |
| Azure AI Search | Knowledge Bases / OpenSearch etc. |
| Azure Key Vault | AWS Secrets Manager |
| Azure Monitor | CloudWatch |
| Azure Functions | Lambda |
| Blob Storage | S3 |

### Interview answer

> "My production experience has primarily been with Amazon Bedrock and Claude. Azure OpenAI follows a similar managed-LLM consumption model, but the surrounding enterprise ecosystem is Azure-native. I would use Azure OpenAI with services such as Azure AI Search, Microsoft Entra ID, Managed Identity, Key Vault, and Azure Monitor. The core AI engineering patterns—RAG, tool calling, prompt engineering, evaluation, observability, and agent orchestration—are transferable between the two platforms."

---

# 20. Enterprise Security

A production Azure OpenAI application can use:

```text
                    Azure OpenAI
                         ▲
                         │
                 Managed Identity
                         │
                   Entra ID / RBAC
                         │
Application ─────── Azure Network
                         │
                    Key Vault
```

Important concepts:

- Microsoft Entra ID
- Managed Identity
- RBAC
- Key Vault
- Private endpoints
- Network isolation
- API security
- Logging/auditing
- Content filtering

Microsoft specifically documents using Managed Identity with Microsoft Entra authentication to avoid storing credentials in applications. 

---

# 21. Production AI Architecture

A production-grade Azure OpenAI application could look like:

```text
                         User
                           │
                           ▼
                    Web / Teams / App
                           │
                           ▼
                    API Management
                           │
                           ▼
                       FastAPI
                           │
                           ▼
                    Agent / LangGraph
                           │
              ┌────────────┼─────────────┐
              │            │             │
              ▼            ▼             ▼
       Azure OpenAI   Azure AI Search   Tools
              │                          │
              │                    REST / DB
              │
              ▼
          Response
```

Supporting infrastructure:

```text
Managed Identity
       │
       ├── Azure OpenAI
       ├── Azure AI Search
       └── Key Vault

Azure Monitor
       │
       ├── Logs
       ├── Metrics
       └── Application Insights
```

---

# 22. Common Interview Questions

### Q1. What is Azure OpenAI?

> Azure OpenAI is Microsoft's managed Azure service for accessing OpenAI models and building enterprise AI applications with Azure-native security, identity, networking, and governance capabilities.

### Q2. What is an Azure OpenAI deployment?

> A deployment is the configured model endpoint through which an application accesses a model. The application typically references the deployment name when making inference requests.

### Q3. Azure OpenAI vs OpenAI API?

> Azure OpenAI provides OpenAI models through Azure with Azure-native enterprise capabilities such as Entra ID, RBAC, Managed Identity, networking, and monitoring.

### Q4. How would you build RAG?

> I would use Azure AI Search as the retrieval layer and Azure OpenAI as the generation layer. Documents are indexed, relevant chunks are retrieved, and those chunks are provided as grounded context to the model.

### Q5. How do you secure Azure OpenAI?

> I would prefer Microsoft Entra ID with Managed Identity, apply RBAC, use Key Vault for secrets where required, restrict network access, and implement monitoring, auditing, and appropriate content-safety controls.

### Q6. How would you integrate Azure OpenAI into LangGraph?

> I would configure Azure OpenAI as the LLM used by the LangGraph nodes. The graph manages state, routing, tool execution, retries, and human-in-the-loop workflows, while Azure OpenAI handles model inference.

---

# 23. Senior-Level Question

### "You have an existing Bedrock + Claude RAG application. How would you move it to Azure?"

A strong answer:

```text
AWS                              Azure

Bedrock + Claude       →      Azure OpenAI
Qdrant                  →      Azure AI Search
S3                      →      Blob Storage
IAM                     →      Entra ID / RBAC
IAM Role                →      Managed Identity
Secrets Manager         →      Key Vault
Lambda                  →      Azure Functions
API Gateway             →      API Management
CloudWatch              →      Azure Monitor
```

Then say:

> "I would not simply replace services one-to-one. I would first identify the functional requirements, model capabilities, retrieval requirements, security requirements, latency, cost, and compliance constraints, then redesign the Azure-native architecture around those requirements."

That is a much stronger **consultant-level answer** than saying "Bedrock is replaced by Azure OpenAI."

---

## Key Interview Takeaways

```text
Azure OpenAI
    │
    ├── Resource
    ├── Deployment
    ├── Models
    ├── Responses API
    ├── Chat Completions
    ├── Tool Calling
    ├── Structured Output
    ├── Streaming
    │
    ├── Authentication
    │      ├── API Key
    │      └── Entra ID / Managed Identity
    │
    ├── RAG
    │      └── Azure AI Search
    │
    └── Production
           ├── Security
           ├── Monitoring
           ├── Scaling
           ├── Cost
           └── Reliability
```

The most current Microsoft documentation recommends the **Responses API for new Azure OpenAI applications**, while Chat Completions remains supported. 